In [ ]:
# ============================================================
# CONFIGURACION DE CREDENCIALES (HIBRIDO: COLAB SECRETS / LOCAL TXT)
# ============================================================
import os

def load_credentials():
    # 1. Intentar cargar desde Google Colab Secrets (si estamos en Colab)
    try:
        from google.colab import userdata
        keys = ['R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY', 'R2_ENDPOINT_URL', 'R2_BUCKET_NAME']
        for k in keys:
            try:
                val = userdata.get(k)
                if val: os.environ[k] = val
            except: pass
    except ImportError:
        pass

    # 2. Si no se cargaron (no estamos en Colab o no hay Secrets), buscar archivo local
    if not os.getenv('R2_ACCESS_KEY_ID'):
        cred_path = '../Keys_and_tokens'
        cred_file = os.path.join(cred_path, 'Credenciales.txt')
        
        if os.path.exists(cred_file):
            with open(cred_file, 'r') as f:
                for line in f:
                    if '=' in line and not line.strip().startswith('#'):
                        parts = line.split('=', 1)
                        key = parts[0].strip()
                        value = parts[1].split('#')[0].strip().strip("'").strip('"')
                        os.environ[key] = value
        else:
            # 3. Si no hay nada, pedir manualmente y guardar
            print(" No se detectaron credenciales (Secrets o TXT).")
            os.makedirs(cred_path, exist_ok=True)
            ak = input("  R2_ACCESS_KEY_ID: ")
            sk = input("  R2_SECRET_ACCESS_KEY: ")
            ep = input("  R2_ENDPOINT_URL: ")
            bn = input("  R2_BUCKET_NAME: ")
            with open(cred_file, 'w') as f:
                f.write(f"R2_ACCESS_KEY_ID = '{ak}'\n")
                f.write(f"R2_SECRET_ACCESS_KEY = '{sk}'\n")
                f.write(f"R2_ENDPOINT_URL = '{ep}'\n")
                f.write(f"R2_BUCKET_NAME = '{bn}'\n")
            os.environ['R2_ACCESS_KEY_ID'] = ak
            os.environ['R2_SECRET_ACCESS_KEY'] = sk
            os.environ['R2_ENDPOINT_URL'] = ep
            os.environ['R2_BUCKET_NAME'] = bn

load_credentials()

R2_ACCESS_KEY_ID     = os.getenv('R2_ACCESS_KEY_ID')
R2_SECRET_ACCESS_KEY = os.getenv('R2_SECRET_ACCESS_KEY')
R2_ENDPOINT_URL      = os.getenv('R2_ENDPOINT_URL')
R2_BUCKET_NAME       = os.getenv('R2_BUCKET_NAME')

CARPETA_DESTINO_R2 = 'estaciones_dagma'
print(f'\n Conectado al Bucket: {R2_BUCKET_NAME}')


# Caracterización del Dataset: Ozono Troposférico ($O_3$)
### Proyecto GeoVision-CLIP Cali | Situación 1

Este notebook presenta la estructura y el muestreo inicial de los datos de Ozono capturados en la red terrestre de Santiago de Cali.

## 1. Ficha Técnica del Dataset
* **Fuente de Datos:** Sistema de Vigilancia de la Calidad del Aire (SVCA) del **DAGMA**.
* **Repositorio Nacional:** SISAIRE (IDEAM).
* **Periodo de Muestreo:** 01 de enero de 2018 al 31 de diciembre de 2022.
* **Resolución Temporal:** Horaria (24 mediciones por día).
* **Unidades:** Microgramos por metro cúbico ($\mu g/m^3$).

## 2. Dinámica del Contaminante
A diferencia de otros gases, el $O_3$ no se emite directamente. Es un **contaminante secundario** que se forma mediante reacciones fotoquímicas:
1. **Precursores:** $NO_2$ (tráfico) y COVs (industria/vegetación).
2. **Catalizador:** Radiación Ultravioleta (Luz Solar).
3. **Hora Crítica:** El pico máximo ocurre entre las **12:00 y las 16:00 COT**, coincidiendo con la máxima insolación y el paso del satélite **Sentinel-5P (13:00 COT aprox.)**.

## 3. Estaciones de Monitoreo (Puntos de Verificación)
El dataset integra información de las estaciones operativas del área metropolitana:
* **Norte/Centro:** La Flora, Ermita, Obrero, Base Aérea.
* **Oriente:** Compartir.
* **Sur:** Univalle, Pance (San Buenaventura).
* **Ladera/Otras:** Siloé, Transitoria.

In [7]:
import plotly.express as px
import pandas as pd
import glob
import os
import zipfile

# 1. Descomprimir archivos
with zipfile.ZipFile('O3DAGMA.zip', 'r') as zip_ref:
    zip_ref.extractall('data_o3')

# 2. Buscar TODOS los CSV sin importar en qué subcarpeta estén escondidos
path = 'data_o3'
all_files = glob.glob(path + "/**/*.csv", recursive=True)

print(f"Archivos encontrados para procesar: {len(all_files)}")

# 3. Leer y concatenar arreglando la codificación (tildes)
df_list = []
for f in all_files:
    # utf-8 arregla el problema de "BASE AÃ‰REA"
    temp_df = pd.read_csv(f, encoding='utf-8')
    df_list.append(temp_df)

df_o3 = pd.concat(df_list, ignore_index=True)

# 4. Limpieza Estructural (Adaptado a tu archivo real)
# Renombramos las columnas para que sean fáciles de manejar
df_o3 = df_o3.rename(columns={
    'Fecha inicial': 'Fecha',
    'O3': 'Concentracion'
})

# Convertimos la fecha a formato de tiempo real
df_o3['Fecha'] = pd.to_datetime(df_o3['Fecha'])

# Forzamos a que la columna de Ozono sea numérica (quita las comillas)
# errors='coerce' convierte cualquier texto raro o vacío en NaN (nulo)
df_o3['Concentracion'] = pd.to_numeric(df_o3['Concentracion'], errors='coerce')

print(f"Dataset de Ozono cargado exitosamente: {df_o3.shape[0]} registros horarios.")
print("\nMuestra de los datos limpios:")
print(df_o3.head())

Archivos encontrados para procesar: 5
Dataset de Ozono cargado exitosamente: 178512 registros horarios.

Muestra de los datos limpios:
     Estacion               Fecha       Fecha final  Concentracion
0  BASE AÉREA 2020-10-15 07:00:00  2020-10-15 07:59         9.5400
1  BASE AÉREA 2020-10-15 06:00:00  2020-10-15 06:59         0.0049
2  BASE AÉREA 2020-10-15 05:00:00  2020-10-15 05:59         0.0049
3  BASE AÉREA 2020-10-15 04:00:00  2020-10-15 04:59         0.0050
4  BASE AÉREA 2020-10-15 03:00:00  2020-10-15 03:59         0.0050


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Análisis de Temporalidad y Calidad del Dato (Data Quality)
Al trabajar con sensores físicos del SVCA (Sistema de Vigilancia de la Calidad del Aire), es imperativo entender la naturaleza de los registros:
* **Valores Faltantes (Huecos):** Los espacios sin datos no son errores de descarga, sino reflejos de la realidad operativa (cortes de energía en la estación, mantenimientos preventivos o fallas de calibración del sensor fotométrico).
* **Efecto Fin de Semana:** Dado que el $NO_2$ (tráfico) es el principal precursor del Ozono, esperamos ver una variación en la amplitud de los picos fotoquímicos durante los sábados y domingos, cuando el parque automotor disminuye.
* **Mapa de Calor Horario:** La mejor forma de visualizar la "receta" del Ozono es mediante una matriz que cruce la hora del día con el día de la semana.

In [8]:
# Preparar datos para el mapa de calor
df_o3['Hora'] = df_o3['Fecha'].dt.hour
df_o3['Dia_Semana'] = df_o3['Fecha'].dt.day_name()

# Ordenar los días lógicamente
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_heatmap = df_o3.groupby(['Dia_Semana', 'Hora'])['Concentracion'].mean().reset_index()

# Crear el Heatmap con Plotly
fig_heat = px.density_heatmap(df_heatmap, x="Hora", y="Dia_Semana", z="Concentracion",
                              category_orders={"Dia_Semana": dias_orden},
                              color_continuous_scale="Viridis",
                              title="<b>Firma Fotoquímica: Concentración Promedio de O3 (Hora vs Día)</b>",
                              labels={"Hora": "Hora del Día (0-23)", "Dia_Semana": "Día", "Concentracion": "O3 (µg/m³)"},
                              template="plotly_dark")

fig_heat.update_layout(yaxis={'categoryorder':'array', 'categoryarray': dias_orden[::-1]})
fig_heat.show()

In [9]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.9 MB/s eta 0:00:00


In [10]:
df_o3.to_parquet('cali_o3_2018_2022.parquet', engine='pyarrow', index=False)